# 🔍 Fraud Detection Intelligence System

This notebook demonstrates the full fraud detection pipeline:
1. Load and explore the dataset
2. Preprocess the data
3. Train multiple ML models (Logistic Regression, Random Forest, XGBoost)
4. Evaluate and compare models
5. Save the best model as `.pkl`
6. Run predictions on new data
7. **Analytics: Anomaly Detection, User Clustering, Spending Score, Association Rules**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Aakash-err404/fraud-detection-intelligence-system/blob/main/notebooks/Fraud_Detection_Intelligence_System.ipynb)

## 1. Install Dependencies

In [ ]:
!pip install -q scikit-learn xgboost imbalanced-learn pandas numpy matplotlib seaborn

## 2. Import Libraries

In [ ]:
import pickle
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
print("Libraries imported successfully!")

## 3. Load Dataset

You can either:
- **Upload your own CSV** via Colab file upload
- **Use the Kaggle Credit Card Fraud dataset** (requires Kaggle API key)
- **Generate a synthetic dataset** for demonstration

In [ ]:
# Option 1: Upload CSV manually
# from google.colab import files
# uploaded = files.upload()
# df = pd.read_csv(list(uploaded.keys())[0])

# Option 2: Download from Kaggle (uncomment and set up API key)
# !pip install -q kaggle
# !mkdir -p ~/.kaggle
# !echo '{"username":"YOUR_USERNAME","key":"YOUR_KEY"}' > ~/.kaggle/kaggle.json
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle datasets download -d mlg-ulb/creditcardfraud --unzip -p data/
# df = pd.read_csv('data/creditcard.csv')

# Option 3: Generate synthetic dataset (default)
def generate_synthetic_fraud_dataset(n_samples=20000, fraud_ratio=0.02):
    """Generate a synthetic dataset mimicking credit card fraud data."""
    X, y = make_classification(
        n_samples=n_samples,
        n_features=28,
        n_informative=15,
        n_redundant=5,
        n_classes=2,
        weights=[1 - fraud_ratio, fraud_ratio],
        flip_y=0.01,
        random_state=42,
    )
    feature_names = [f"V{i}" for i in range(1, 29)]
    df = pd.DataFrame(X, columns=feature_names)
    rng = np.random.RandomState(42)
    df.insert(0, "Time", np.sort(rng.uniform(0, 172800, n_samples)))
    df["Amount"] = np.abs(rng.lognormal(3, 2, n_samples))
    df["Class"] = y
    return df

df = generate_synthetic_fraud_dataset()
print(f"Dataset shape: {df.shape}")
df.head()

## 4. Exploratory Data Analysis

In [ ]:
print("Dataset Info:")
print(f"  Shape: {df.shape}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"\nClass Distribution:")
print(df["Class"].value_counts())
print(f"\nFraud ratio: {df['Class'].mean():.4f} ({df['Class'].mean()*100:.2f}%)")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Class distribution
class_counts = df["Class"].value_counts()
axes[0].bar(["Not Fraud", "Fraud"], class_counts.values, color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Class Distribution")
axes[0].set_ylabel("Count")

# Amount distribution by class
df[df["Class"] == 0]["Amount"].hist(bins=50, ax=axes[1], alpha=0.7, label="Not Fraud", color="#2ecc71")
df[df["Class"] == 1]["Amount"].hist(bins=50, ax=axes[1], alpha=0.7, label="Fraud", color="#e74c3c")
axes[1].set_title("Transaction Amount Distribution")
axes[1].legend()

# Correlation heatmap (top features)
corr = df.corr()["Class"].drop("Class").abs().sort_values(ascending=False)
top_features = corr.head(10)
axes[2].barh(top_features.index, top_features.values, color="steelblue")
axes[2].set_title("Top 10 Features Correlated with Fraud")
axes[2].set_xlabel("|Correlation|")

plt.tight_layout()
plt.show()

## 5. Preprocessing

In [ ]:
TARGET_COL = "Class"

# Separate features and target
feature_cols = [c for c in df.columns if c != TARGET_COL]
numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()

X = df[feature_cols]
y = df[TARGET_COL].astype(int)

print(f"Features: {len(feature_cols)} ({len(numeric_cols)} numeric, {len(categorical_cols)} categorical)")
print(f"Target: {TARGET_COL}")
print(f"Class distribution: {dict(y.value_counts())}")

In [ ]:
# Build preprocessing pipeline
transformers = []
if numeric_cols:
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    transformers.append(("num", numeric_pipeline, numeric_cols))

if categorical_cols:
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])
    transformers.append(("cat", categorical_pipeline, categorical_cols))

preprocessor = ColumnTransformer(transformers=transformers, remainder="drop")

# Train-test split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y,
)
print(f"Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples")
print(f"Train fraud ratio: {y_train.mean():.4f} | Test fraud ratio: {y_test.mean():.4f}")

## 6. Model Training

We train three models and compare their performance:
1. **Logistic Regression** — fast baseline
2. **Random Forest** — ensemble method
3. **XGBoost** — gradient boosting

In [ ]:
def compute_class_weights(y):
    counts = y.value_counts()
    total = len(y)
    return {cls: total / (len(counts) * count) for cls, count in counts.items()}

class_weights = compute_class_weights(y_train)
scale_pos_weight = max(class_weights.get(1, 1) / class_weights.get(0, 1), 1.0)

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, max_depth=15, class_weight="balanced",
        random_state=42, n_jobs=-1,
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=scale_pos_weight, random_state=42,
        eval_metric="logloss", n_jobs=-1,
    ),
}

results = {}
smote = SMOTE(random_state=42)

for name, classifier in models.items():
    print(f"\nTraining {name}...")
    pipeline = ImbPipeline([
        ("preprocessor", preprocessor),
        ("smote", smote),
        ("classifier", classifier),
    ])
    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_proba = pipeline.predict_proba(X_test)[:, 1] if hasattr(pipeline, "predict_proba") else None

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1_score": f1_score(y_test, y_pred, zero_division=0),
    }
    cm = confusion_matrix(y_test, y_pred)

    results[name] = {
        "pipeline": pipeline,
        "metrics": metrics,
        "confusion_matrix": cm,
        "y_pred": y_pred,
        "y_proba": y_proba,
    }

    print(f"  Accuracy:  {metrics['accuracy']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall:    {metrics['recall']:.4f}")
    print(f"  F1 Score:  {metrics['f1_score']:.4f}")

## 7. Model Comparison

In [ ]:
# Metrics comparison table
comparison_df = pd.DataFrame(
    {name: res["metrics"] for name, res in results.items()}
).T
comparison_df = comparison_df.round(4)
print("Model Comparison:")
display(comparison_df)

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (name, res) in zip(axes, results.items()):
    sns.heatmap(
        res["confusion_matrix"], annot=True, fmt="d", cmap="Blues",
        xticklabels=["Not Fraud", "Fraud"],
        yticklabels=["Not Fraud", "Fraud"],
        ax=ax,
    )
    ax.set_title(f"{name}")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.tight_layout()
plt.show()

In [ ]:
# Metrics bar chart
fig, ax = plt.subplots(figsize=(10, 6))
comparison_df.plot(kind="bar", ax=ax)
ax.set_title("Model Comparison")
ax.set_ylabel("Score")
ax.set_ylim(0, 1.1)
ax.legend(loc="lower right")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. Feature Importance (Random Forest)

In [ ]:
rf_pipeline = results["Random Forest"]["pipeline"]
rf_classifier = rf_pipeline.named_steps["classifier"]

# Get feature names from preprocessor
fitted_preprocessor = rf_pipeline.named_steps["preprocessor"]
feature_names_out = []
for name, transformer, columns in fitted_preprocessor.transformers_:
    if name == "num":
        feature_names_out.extend(columns)
    elif name == "cat":
        encoder = transformer.named_steps.get("encoder")
        if encoder and hasattr(encoder, "get_feature_names_out"):
            feature_names_out.extend(encoder.get_feature_names_out(columns))
        else:
            feature_names_out.extend(columns)

importances = rf_classifier.feature_importances_
indices = np.argsort(importances)[-15:]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(indices)), importances[indices], align="center", color="steelblue")
ax.set_yticks(range(len(indices)))
ax.set_yticklabels([feature_names_out[i] for i in indices])
ax.set_xlabel("Importance")
ax.set_title("Top 15 Feature Importances (Random Forest)")
plt.tight_layout()
plt.show()

## 9. Save Best Model

In [ ]:
# Select best model by F1 score
best_name = max(results, key=lambda k: results[k]["metrics"]["f1_score"])
best_result = results[best_name]
print(f"Best model: {best_name} (F1: {best_result['metrics']['f1_score']:.4f})")

# Save model artifact
artifact = {
    "pipeline": best_result["pipeline"],
    "model_name": best_name,
    "target_col": TARGET_COL,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "metrics": best_result["metrics"],
}

model_path = "saved_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(artifact, f)

import os
print(f"Model saved to: {model_path}")
print(f"File size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB")

## 10. Demonstrate Predictions

In [ ]:
# Load saved model
with open(model_path, "rb") as f:
    loaded_artifact = pickle.load(f)

loaded_pipeline = loaded_artifact["pipeline"]

# Run predictions on test set
sample_data = X_test.head(20).copy()
predictions = loaded_pipeline.predict(sample_data)
probabilities = loaded_pipeline.predict_proba(sample_data)[:, 1]

# Build results table
pred_df = sample_data.copy()
pred_df["Fraud_Probability"] = np.round(probabilities, 4)
pred_df["Prediction"] = predictions
pred_df["Prediction_Label"] = pred_df["Prediction"].map({0: "Not Fraud", 1: "Fraud"})
pred_df["Actual"] = y_test.head(20).values
pred_df["Actual_Label"] = pred_df["Actual"].map({0: "Not Fraud", 1: "Fraud"})

display_cols = ["Prediction_Label", "Fraud_Probability", "Actual_Label", "Amount"]
display(pred_df[display_cols])

In [ ]:
# Classification report for the best model
print(f"\nClassification Report ({best_name}):")
print(classification_report(
    y_test, best_result["y_pred"],
    target_names=["Not Fraud", "Fraud"],
    zero_division=0,
))

## 11. Analytics — Data Mining Techniques (Case Study)

The following sections demonstrate the five data-mining techniques from the case study:
1. **Anomaly Detection** — Isolation Forest
2. **User Clustering** — K-Means segmentation
3. **Spending Score** — Weighted regression
4. **Rule-based Fraud Indicators** — Heuristic rules
5. **Association Rule Mining** — Apriori algorithm

### 11a. Anomaly Detection (Isolation Forest)

Isolation Forest identifies transactions that are statistically unusual. Anomalies are data points that are few and different — they are easier to "isolate" in random tree partitions.

In [ ]:
from sklearn.ensemble import IsolationForest

# Use numeric features (excluding target)
anomaly_features = [c for c in numeric_cols if c != TARGET_COL]
X_anomaly = df[anomaly_features].copy().fillna(df[anomaly_features].median())

scaler_anom = StandardScaler()
X_scaled = scaler_anom.fit_transform(X_anomaly)

iso_forest = IsolationForest(contamination=0.05, random_state=42, n_jobs=-1)
anomaly_labels = iso_forest.fit_predict(X_scaled)
anomaly_scores = iso_forest.decision_function(X_scaled)

df_anomaly = df.copy()
df_anomaly["Anomaly"] = anomaly_labels
df_anomaly["Anomaly_Score"] = anomaly_scores

anom_count = (anomaly_labels == -1).sum()
print(f"Anomalies detected: {anom_count} / {len(df)} ({anom_count/len(df)*100:.1f}%)")

# Scatter plot
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
normal = df_anomaly[df_anomaly["Anomaly"] == 1]
anomaly = df_anomaly[df_anomaly["Anomaly"] == -1]

axes[0].scatter(normal[anomaly_features[0]], normal["Amount"], c="#2ecc71", alpha=0.4, s=10, label="Normal")
axes[0].scatter(anomaly[anomaly_features[0]], anomaly["Amount"], c="#e74c3c", alpha=0.8, s=30, marker="x", label="Anomaly")
axes[0].set_xlabel(anomaly_features[0])
axes[0].set_ylabel("Amount")
axes[0].set_title("Anomaly Detection Results")
axes[0].legend()

axes[1].hist(anomaly_scores, bins=50, color="steelblue", edgecolor="white", alpha=0.8)
axes[1].axvline(x=0, color="red", linestyle="--", label="Decision boundary")
axes[1].set_xlabel("Anomaly Score")
axes[1].set_ylabel("Count")
axes[1].set_title("Distribution of Anomaly Scores")
axes[1].legend()

plt.tight_layout()
plt.show()

### 11b. User Clustering (K-Means Segmentation)

K-Means groups transactions into segments based on spending patterns:
- **Low Spenders** — low amount, low activity
- **Regular Users** — moderate usage
- **High Spenders** — high amount, high frequency (need monitoring)

In [ ]:
from sklearn.cluster import KMeans

cluster_features = [c for c in numeric_cols if c != TARGET_COL]
X_cluster = df[cluster_features].copy().fillna(df[cluster_features].median())

scaler_clust = StandardScaler()
X_clust_scaled = scaler_clust.fit_transform(X_cluster)

km = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = km.fit_predict(X_clust_scaled)

df_cluster = df.copy()
df_cluster["Cluster"] = cluster_labels

# Label clusters by mean Amount
cluster_means = df_cluster.groupby("Cluster")["Amount"].mean()
order = cluster_means.sort_values().index.tolist()
segment_names = ["Low Spenders", "Regular Users", "High Spenders"]
label_map = {old: segment_names[i] for i, old in enumerate(order)}
df_cluster["Cluster_Label"] = df_cluster["Cluster"].map(label_map)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Cluster distribution
counts = df_cluster["Cluster_Label"].value_counts().reindex(segment_names)
colors_clust = ["#2ecc71", "#f39c12", "#e74c3c"]
axes[0].bar(counts.index, counts.values, color=colors_clust)
axes[0].set_title("Cluster Distribution")
axes[0].set_ylabel("Count")

# Scatter plot
palette = sns.color_palette("husl", 3)
for cid in sorted(df_cluster["Cluster"].unique()):
    subset = df_cluster[df_cluster["Cluster"] == cid]
    lbl = subset["Cluster_Label"].iloc[0]
    axes[1].scatter(subset[cluster_features[0]], subset["Amount"], color=palette[cid], alpha=0.5, s=15, label=lbl)
axes[1].set_xlabel(cluster_features[0])
axes[1].set_ylabel("Amount")
axes[1].set_title("User Segmentation (K-Means)")
axes[1].legend(title="Segment")

plt.tight_layout()
plt.show()

print("\nCluster Summary:")
display(df_cluster.groupby("Cluster_Label")[cluster_features].mean().round(4))

### 11c. Spending Score (Regression)

A weighted spending score per the case-study formula:

```
Score = 0.3 × Amount + 0.2 × Frequency + 0.3 × Time + 0.2 × Category
```

Features are min-max scaled to [0, 1] before weighting. The score classifies users into:
- **Normal User** (low score)
- **Active User** (medium score)
- **High Value / Risky** (high score)

In [ ]:
def min_max_scale(series):
    mn, mx = series.min(), series.max()
    if mx == mn:
        return pd.Series(0.5, index=series.index)
    return (series - mn) / (mx - mn)

# Compute spending score using available features
score = pd.Series(0.0, index=df.index)
weights = {"amount": 0.3, "time": 0.3}  # frequency and category not in synthetic data

if "Amount" in df.columns:
    score += weights["amount"] * min_max_scale(df["Amount"].fillna(0))
if "Time" in df.columns:
    score += weights["time"] * min_max_scale(df["Time"].fillna(0))

# Use first two V-features as proxy for remaining weight
remaining_weight = 1.0 - sum(weights.values())
proxy_cols = [c for c in numeric_cols if c.startswith("V")][:2]
for i, c in enumerate(proxy_cols):
    score += (remaining_weight / len(proxy_cols)) * min_max_scale(df[c].fillna(0))

df_score = df.copy()
df_score["Spending_Score"] = score

q33 = score.quantile(0.33)
q66 = score.quantile(0.66)
df_score["Spending_Segment"] = pd.cut(
    score,
    bins=[-np.inf, q33, q66, np.inf],
    labels=["Normal User", "Active User", "High Value / Risky"],
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(score, bins=40, color="steelblue", edgecolor="white", alpha=0.8)
axes[0].axvline(x=q33, color="orange", linestyle="--", label=f"33rd pctl ({q33:.3f})")
axes[0].axvline(x=q66, color="red", linestyle="--", label=f"66th pctl ({q66:.3f})")
axes[0].set_xlabel("Spending Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Spending Score Distribution")
axes[0].legend()

seg_counts = df_score["Spending_Segment"].value_counts()
seg_order = ["Normal User", "Active User", "High Value / Risky"]
seg_counts = seg_counts.reindex([o for o in seg_order if o in seg_counts.index])
axes[1].bar(seg_counts.index, seg_counts.values, color=["#2ecc71", "#f39c12", "#e74c3c"][:len(seg_counts)])
axes[1].set_ylabel("Count")
axes[1].set_title("Spending Segments")

plt.tight_layout()
plt.show()

print(f"\nSpending Score stats: mean={score.mean():.4f}, std={score.std():.4f}")
display(df_score.groupby("Spending_Segment")[["Amount", "Time"]].mean().round(2))

### 11d. Rule-based Fraud Indicators

Heuristic rules from the case study that flag suspicious transactions:
- **R1**: Amount > 50,000 AND late night (0–4 AM) → Fraud
- **R2**: Rare/new location + elevated amount → Fraud
- **R3**: High transaction frequency → Fraud

In [ ]:
# Apply rule-based fraud flags
df_rules = df.copy()
flags = pd.Series(False, index=df.index)
reasons_list = [[] for _ in range(len(df))]

# Rule 1: High amount (using Amount column, >95th percentile as threshold)
amount_threshold = df["Amount"].quantile(0.95)
high_amount = df["Amount"] > amount_threshold
flags |= high_amount
for idx in high_amount[high_amount].index:
    reasons_list[idx].append(f"Amount > {amount_threshold:.0f} (95th percentile)")

# Rule 2: High amount + unusual Time (late night proxy: Time in first 4 hours)
if "Time" in df.columns:
    time_threshold = 4 * 3600  # first 4 hours in seconds
    late_night = df["Time"] < time_threshold
    mask = high_amount & late_night
    for idx in mask[mask].index:
        if "late night" not in str(reasons_list[idx]):
            reasons_list[idx].append("High amount + late night")

df_rules["Rule_Flag"] = flags
df_rules["Rule_Reasons"] = ["; ".join(r) if r else "" for r in reasons_list]

flagged_count = flags.sum()
print(f"Flagged transactions: {flagged_count} / {len(df)} ({flagged_count/len(df)*100:.1f}%)")
print(f"\nSample flagged transactions:")
display(df_rules[df_rules["Rule_Flag"]][["Amount", "Time", "Rule_Reasons", "Class"]].head(10))

### 11e. Association Rule Mining (Apriori)

Discover hidden patterns in transactions. Numeric features are discretized into Low / Med / High bins.

Example rules from the case study:
- Night + High Amount → Fraud
- Travel + High Amount → Premium Users
- Food + Evening → Frequent

In [ ]:
!pip install -q mlxtend

from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

# Use a subset of features for association mining
assoc_cols = ["Amount", "Time", "Class"]
n_bins = 3
bin_labels = ["Low", "Med", "High"]

items_per_row = []
for _, row in df[assoc_cols].head(5000).iterrows():  # limit for speed
    items = []
    for col in assoc_cols:
        val = row[col]
        if pd.isna(val):
            continue
        if isinstance(val, (int, float, np.integer, np.floating)):
            try:
                bin_label = pd.cut([val], bins=n_bins, labels=bin_labels)[0]
            except Exception:
                bin_label = "Med"
            items.append(f"{col}={bin_label}")
        else:
            items.append(f"{col}={val}")
    items_per_row.append(items)

te = TransactionEncoder()
te_array = te.fit(items_per_row).transform(items_per_row)
basket = pd.DataFrame(te_array, columns=te.columns_)

freq_items = apriori(basket, min_support=0.05, use_colnames=True)
if not freq_items.empty:
    rules = association_rules(freq_items, metric="confidence", min_threshold=0.5)
    rules["antecedents"] = rules["antecedents"].apply(lambda x: ", ".join(sorted(x)))
    rules["consequents"] = rules["consequents"].apply(lambda x: ", ".join(sorted(x)))
    rules_display = rules[["antecedents", "consequents", "support", "confidence", "lift"]].sort_values("lift", ascending=False)
    print(f"Found {len(rules_display)} association rules:")
    display(rules_display.head(20))
else:
    print("No frequent itemsets found with current thresholds.")

In [ ]:
# Download the model (for Google Colab)
# from google.colab import files
# files.download(model_path)
print("Done! The model can be used with the Streamlit app.")